# Spectral Rank Under Sum vs Mean Aggregation

**Validating the Information-Compression Mechanism at FK Joins**

This notebook demonstrates the spectral rank analysis experiment comparing **SUM** vs **MEAN** aggregation on RelBench tasks. The experiment:

1. Trains HeteroSAGE with both sum and mean aggregation on 3 RelBench tasks
2. Instruments the aggregation step to capture child embeddings at FK joins
3. Computes SVD-based effective rank and variance-entropy proxy metrics
4. Compares rank preservation between aggregation types

**Core hypothesis**: Sum aggregation preserves higher effective spectral rank than mean at FK joins, explaining both sum's performance advantage AND why CAMA helps mean but not sum/RelGNN.

The demo loads pre-computed results and visualizes the rank comparison across tasks and edge types.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# All packages used are pre-installed on Colab; install locally to match Colab env
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'matplotlib==3.10.0')
    _pip('scipy==1.15.3')  # Colab: 1.16.3; 1.15.3 is latest for Python 3.10
    _pip('torch==2.9.0', '--index-url', 'https://download.pytorch.org/whl/cpu')

In [ ]:
import json
import math
import os
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import torch
from scipy.stats import spearmanr
import matplotlib.pyplot as plt

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-outputs/ai-invention-9b0386-rank-aware-moment-aggregation-diversity-/main/experiment_iter6_spectral_rank_u/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print(f"Loaded data with {len(data['datasets'])} datasets")
print(f"Metadata title: {data['metadata']['title']}")
print(f"Hypothesis: {data['metadata']['hypothesis_tested']}")

In [ ]:
# ── Configuration ──
# Tunable parameters for the synthetic smoke test demonstration
SEED = 42
EPS = 1e-10
MAX_CHILDREN_FOR_SVD = 500   # Cap children for SVD performance
# Smoke test parameters
N_GROUPS = 5        # Number of parent groups in smoke test
N_PER_GROUP = 4     # Children per group
DIM = 8             # Embedding dimension

# CAMA effect sizes from the original experiment
CAMA_COHENS_D = {
    "rel-f1/driver-dnf": 1.13,
    "rel-trial/study-adverse": -2.45,
    "rel-stack/user-engagement": 7.95,
}

## Rank Computation Functions

These functions compute the spectral rank metrics used to measure information preservation at FK joins:

- **SVD Effective Rank**: `erank = exp(H(σ))` where σ are normalized singular values — measures the "effective dimensionality" of the child embedding matrix
- **Variance-Entropy Proxy**: `H(var) / log(d)` — a cheaper proxy for rank diversity, normalized to [0, 1]
- **Compression Ratio**: `erank / min(N, d)` — ratio of effective rank to maximum possible rank

In [ ]:
def compute_svd_effective_rank(C: torch.Tensor, eps: float = EPS) -> float:
    """SVD-based effective rank of child embedding matrix C [N, d].

    erank = exp(H(sigma)) where sigma are normalized singular values.
    """
    if C.shape[0] < 2:
        return 1.0
    # Cap children for SVD performance
    if C.shape[0] > MAX_CHILDREN_FOR_SVD:
        perm = torch.randperm(C.shape[0])[:MAX_CHILDREN_FOR_SVD]
        C = C[perm]
    C = C.float()
    try:
        s = torch.linalg.svdvals(C)
    except Exception:
        return float("nan")
    s = s[s > eps]
    if len(s) == 0:
        return 0.0
    p = s / s.sum()
    H = -(p * p.log()).sum().item()
    return math.exp(H)


def compute_variance_entropy_proxy(
    C: torch.Tensor, d: int, eps: float = EPS
) -> float:
    """Variance-entropy proxy: H(var) / log(d), normalized to [0, 1].

    Matches RAMA blueprint exactly.
    """
    if C.shape[0] < 2:
        return 1.0 / max(d, 1)
    var = C.var(dim=0)
    var_sum = var.sum()
    if var_sum < eps:
        return 1.0 / max(d, 1)
    p = var / var_sum
    p = p.clamp(min=eps)
    H = -(p * p.log()).sum().item()
    log_d = math.log(max(d, 2))
    return H / log_d


def compute_compression_ratio(erank: float, N: int, d: int) -> float:
    """Ratio of effective rank to maximum possible rank = min(N, d)."""
    max_rank = min(N, d)
    if max_rank == 0:
        return float("nan")
    return erank / max_rank

## Smoke Test: Rank Functions on Synthetic Data

Verify the rank computation functions work correctly on synthetic child embeddings. We create groups of child vectors and compute their spectral rank under sum vs mean aggregation.

In [ ]:
torch.manual_seed(SEED)
N_total = N_GROUPS * N_PER_GROUP
x = torch.randn(N_total, DIM)
index = torch.arange(N_GROUPS).repeat_interleave(N_PER_GROUP)

print(f"Synthetic data: {N_total} child vectors of dim {DIM}, assigned to {N_GROUPS} parent groups")
print(f"Index: {index.tolist()}\n")

# Compute sum and mean aggregation outputs using plain torch
sum_out = torch.zeros(N_GROUPS, DIM)
mean_out = torch.zeros(N_GROUPS, DIM)
for g in range(N_GROUPS):
    mask = index == g
    sum_out[g] = x[mask].sum(dim=0)
    mean_out[g] = x[mask].mean(dim=0)

# Verify shapes
assert sum_out.shape == (N_GROUPS, DIM), f"Expected ({N_GROUPS},{DIM}), got {sum_out.shape}"
assert mean_out.shape == (N_GROUPS, DIM), f"Expected ({N_GROUPS},{DIM}), got {mean_out.shape}"

# Compute rank metrics per group
print(f"{'Group':>6} | {'N':>3} | {'SVD Rank':>10} | {'Proxy':>8} | {'Compress':>10}")
print("-" * 55)
for g in range(N_GROUPS):
    mask = index == g
    C = x[mask]
    N = C.shape[0]
    svd_rank = compute_svd_effective_rank(C)
    proxy = compute_variance_entropy_proxy(C, DIM)
    compression = compute_compression_ratio(svd_rank, N, DIM)
    print(f"{g:>6} | {N:>3} | {svd_rank:>10.4f} | {proxy:>8.4f} | {compression:>10.4f}")

print("\nSmoke test PASSED: rank functions work correctly on synthetic data")

## Parse Pre-computed Results

Extract per-edge-type rank metrics from the pre-computed experiment data, separated by aggregation type and task.

In [ ]:
# Parse datasets into structured records
records = []
for ds in data["datasets"]:
    ds_name = ds["dataset"]
    for ex in ds["examples"]:
        inp = json.loads(ex["input"])
        out = json.loads(ex["output"])
        records.append({**inp, **out, "dataset_name": ds_name})

print(f"Parsed {len(records)} edge-type measurements\n")

# Separate by aggregation type
mean_records = [r for r in records if r.get("aggr_type") == "mean"]
sum_records = [r for r in records if r.get("aggr_type") == "sum"]
cross_records = [r for r in records if r.get("analysis_type") == "sum_vs_mean_rank_comparison"]

print(f"Mean aggregation records: {len(mean_records)}")
print(f"Sum aggregation records:  {len(sum_records)}")
print(f"Cross-comparison records: {len(cross_records)}")

# Show a sample record
print("\nSample mean-aggregation record:")
if mean_records:
    sample = mean_records[0]
    for k, v in sample.items():
        if not isinstance(v, dict):
            print(f"  {k}: {v}")

## Cross-Aggregation Comparison

Compare spectral rank between sum and mean aggregation per task. The key metric is the **SVD rank ratio** (sum_rank / mean_rank): values > 1 indicate sum preserves more rank.

In [ ]:
# Display cross-aggregation comparison results
print("Cross-Aggregation Comparison: Sum vs Mean Rank Ratios")
print("=" * 75)
print(f"{'Task':<35} | {'Rank Ratio':>11} | {'Compr Diff':>11} | {'Sum Higher':>11}")
print("-" * 75)

for rec in cross_records:
    task = rec["task"]
    ratio = rec.get("mean_svd_rank_ratio")
    comp_diff = rec.get("mean_compression_diff")
    frac = rec.get("fraction_sum_higher_rank")
    print(f"{task:<35} | {ratio:>11.4f} | {comp_diff:>+11.4f} | {frac:>10.1%}")

# Overall comparison from metadata
overall = data["metadata"]["overall_comparison"]
print("-" * 75)
print(f"{'Grand Mean':<35} | {overall['grand_mean_rank_ratio']:>11.4f} | "
      f"{overall['grand_mean_compression_diff']:>+11.4f} |")
print(f"\nHypothesis supported: {overall['hypothesis_supported']}")
print(f"Interpretation: {overall['interpretation']}")

## Per-Edge-Type Metrics Table

Show detailed metrics for each edge type under both aggregation conditions, focusing on non-trivial edges (cardinality > 1).

In [ ]:
# Build per-task tables of non-trivial edges (cardinality > 1)
tasks = sorted(set(r["task"] for r in mean_records))

for task in tasks:
    task_mean = [r for r in mean_records if r["task"] == task and r.get("mean_cardinality", 0) > 1]
    task_sum = [r for r in sum_records if r["task"] == task and r.get("mean_cardinality", 0) > 1]

    if not task_mean and not task_sum:
        continue

    print(f"\n{'='*70}")
    print(f"Task: {task} (CAMA Cohen's d = {CAMA_COHENS_D.get(task, 'N/A')})")
    print(f"{'='*70}")
    print(f"{'Aggr':<6} | {'Edge Type':<40} | {'SVD Rank':>9} | {'Proxy':>7} | {'Card':>6}")
    print("-" * 78)

    for r in task_mean:
        edge_short = r.get("edge_description", r.get("edge_type", "?"))[:40]
        print(f"{'mean':<6} | {edge_short:<40} | {r.get('mean_svd_rank', 0):>9.3f} | "
              f"{r.get('mean_proxy_rank', 0):>7.4f} | {r.get('mean_cardinality', 0):>6.1f}")
    for r in task_sum:
        edge_short = r.get("edge_description", r.get("edge_type", "?"))[:40]
        print(f"{'sum':<6} | {edge_short:<40} | {r.get('mean_svd_rank', 0):>9.3f} | "
              f"{r.get('mean_proxy_rank', 0):>7.4f} | {r.get('mean_cardinality', 0):>6.1f}")

## Visualization: Rank Ratio and Compression Difference Across Tasks

Bar charts showing (1) the SVD rank ratio (sum/mean) per task, and (2) the compression difference (sum − mean) per task. A rank ratio > 1 or positive compression difference would support the hypothesis.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# ── Panel 1: SVD Rank Ratio per task ──
task_labels = [r["task"].replace("/", "\n") for r in cross_records]
rank_ratios = [r["mean_svd_rank_ratio"] for r in cross_records]

colors = ['#e74c3c' if v < 1.0 else '#2ecc71' for v in rank_ratios]
bars = axes[0].bar(task_labels, rank_ratios, color=colors, edgecolor='black', linewidth=0.8)
axes[0].axhline(y=1.0, color='black', linestyle='--', linewidth=1, label='Equal rank (ratio=1)')
axes[0].set_ylabel('SVD Rank Ratio (sum/mean)')
axes[0].set_title('SVD Rank Ratio per Task')
axes[0].legend(fontsize=8)
for bar, val in zip(bars, rank_ratios):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'{val:.3f}', ha='center', va='bottom', fontsize=9)

# ── Panel 2: Compression Difference per task ──
comp_diffs = [r["mean_compression_diff"] for r in cross_records]
colors2 = ['#e74c3c' if v < 0 else '#2ecc71' for v in comp_diffs]
bars2 = axes[1].bar(task_labels, comp_diffs, color=colors2, edgecolor='black', linewidth=0.8)
axes[1].axhline(y=0, color='black', linestyle='--', linewidth=1)
axes[1].set_ylabel('Compression Diff (sum − mean)')
axes[1].set_title('Compression Difference per Task')
for bar, val in zip(bars2, comp_diffs):
    offset = 0.001 if val >= 0 else -0.003
    axes[1].text(bar.get_x() + bar.get_width()/2, val + offset,
                 f'{val:+.4f}', ha='center', va='bottom' if val >= 0 else 'top', fontsize=9)

# ── Panel 3: Fraction of edges where sum has higher rank ──
fractions = [r["fraction_sum_higher_rank"] for r in cross_records]
colors3 = ['#e74c3c' if v < 0.5 else '#2ecc71' for v in fractions]
bars3 = axes[2].bar(task_labels, fractions, color=colors3, edgecolor='black', linewidth=0.8)
axes[2].axhline(y=0.5, color='black', linestyle='--', linewidth=1, label='50% threshold')
axes[2].set_ylabel('Fraction of Edges')
axes[2].set_title('Fraction of Edges Where Sum > Mean Rank')
axes[2].set_ylim(0, 1)
axes[2].legend(fontsize=8)
for bar, val in zip(bars3, fractions):
    axes[2].text(bar.get_x() + bar.get_width()/2, val + 0.02,
                 f'{val:.1%}', ha='center', va='bottom', fontsize=9)

plt.suptitle('Spectral Rank: Sum vs Mean Aggregation Comparison', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('rank_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved to rank_comparison.png")

## Summary

In [ ]:
# Final summary
print("=" * 60)
print("EXPERIMENT SUMMARY")
print("=" * 60)
print(f"Model: {data['metadata']['configuration']['model']}")
print(f"Channels: {data['metadata']['configuration']['channels']}")
print(f"Layers: {data['metadata']['configuration']['num_layers']}")
print(f"Epochs: {data['metadata']['configuration']['epochs']}")
print(f"Tasks compared: {overall['num_tasks_compared']}")
print()
print("KEY FINDINGS:")
findings = data["metadata"]["key_findings"]
print(f"  Grand mean rank ratio (sum/mean): {findings['grand_mean_rank_ratio']:.4f}")
print(f"  Grand mean compression diff:      {findings['grand_mean_compression_diff']:+.4f}")
print(f"  Sum preserves more rank:           {findings['sum_preserves_more_rank']}")
print()
print("CONCLUSION:")
print(f"  {overall['interpretation']}")